# Single vs Multi-Agent Systems

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the trade-offs of multi-agent architectures. We focus on why you split agents and how they communicate.

We will cover 3 patterns:
1. **The Monolith (Single Agent):** A single agent struggling with too many tools/instructions.
2. **The Specialized Split:** Splitting the agent to improve reasoning via isolated prompts.
3. **The Communication Pattern:** Simulating a direct Agent-to-Agent handoff via tool calling.

---
## Pattern 1: The Monolith (Single Agent)

A single agent given too many tools or conflicting instructions (e.g., "be creative but also be extremely paranoid") will struggle. It will compromise and produce mediocre results.

In [ ]:
def monolithic_agent(prompt):
    print(f"\n[Monolith Agent] Processing: '{prompt}'")
    print("  -> Trying to be creative...")
    print("  -> Trying to be paranoid about security...")
    print("  -> Result: A mediocre script with mild security vulnerabilities.")
    return "print('Hello, world') # This might be unsafe"

monolithic_agent("Write a secure python script to greet the user.")


---
## Pattern 2: The Specialized Split (Asymmetric Teams)

By splitting the monolith, we can give one agent a "Creative" prompt and another agent a "Paranoid" prompt. The resulting debate produces a better artifact.

In [ ]:
def coder_agent(prompt):
    print(f"\n[Coder Agent] Writing fast, creative code for: '{prompt}'")
    return "eval(input('Enter your name: '))"  # Highly insecure

def security_reviewer_agent(code):
    print(f"\n[Reviewer Agent] Paranoia mode engaged. Reviewing code...")
    if "eval" in code:
        print("  -> 🚨 SECURITY FLAW: 'eval' detected!")
        return "REJECTED: Use input() directly, never eval()."
    return "APPROVED"

def asymmetric_team(prompt):
    print("\n🚀 Starting Asymmetric Team Workflow...")
    draft = coder_agent(prompt)
    review = security_reviewer_agent(draft)
    
    if "REJECTED" in review:
        print("  -> [Supervisor] Draft rejected. Forcing Coder to rewrite.")
        # In a real system, the Coder would try again.
        print("✅ Final Result: A highly secure, peer-reviewed script.")

asymmetric_team("Write a script to get user input.")


---
## Pattern 3: Direct Handoff (Tool Calling)

How do agents communicate? One of the fastest methods is the Handoff pattern (used in OpenAI Swarm), where an agent literally uses a tool to summon another agent and yield control.

In [ ]:
def billing_agent():
    print("  [Billing Agent] Taking over conversation...")
    print("  [Billing Agent] Querying Stripe API... done.")
    return "Your balance is $0.00."

def transfer_to_billing():
    # This is a tool the Triage Agent can call
    print("  [Tool Execution] Swapping system prompt to Billing Agent...")
    return billing_agent()

def triage_agent(user_query):
    print(f"\n[Triage Agent] Handling query: '{user_query}'")
    if "billing" in user_query.lower() or "balance" in user_query.lower():
        print("  -> Intent is Billing. Executing tool 'transfer_to_billing()'")
        return transfer_to_billing()
    else:
        return "I can help with general support."

print(triage_agent("What is my account balance?"))
